# functime on the StarCraft II snapshot

[functime](https://docs.functime.ai/) does time series forecasting and feature extraction on
[polars](https://pola.rs/) data frames. This notebook uses it on the snapshot written by
s2protocol-rs `write-arrow-ipc` (or swarmy's Optimize), from the simplest to the most useful:

1. **Panel data**: the format functime expects, entity / time / value.
2. **Forecasting the race balance**: weekly win rates per matchup, baselines vs autoregressive
   models, evaluation and cross-validation.
3. **Feature extraction on long series**: is an upgrade researched earlier over the months?
4. **A global model across games**: forecasting the workers of minutes 7 to 10 from minutes 1 to 6,
   one model learns from thousands of games.
5. **Features of the in-game curves**: how the army of the winners grows compared to the losers.

The snapshot is partitioned by day (`ipcs/year=*/month=*/day=*/<file>.arrow`), `snapshot.py` next
to this notebook reads it, see its docstrings. The outputs were generated on a snapshot of 36,151
replays from 2020 to 2026.

In [1]:
import warnings

# flaml, a functime dependency, warns that its optional AutoML extras are not installed. Only
# that warning is ignored, the warnings of polars name the change to make and are kept.
warnings.filterwarnings("ignore", message="flaml.automl is not available")

import polars as pl

import snapshot as s2
from functime.cross_validation import expanding_window_split, train_test_split
from functime.evaluation import rank_point_forecasts
from functime.forecasting import knn, linear_model, naive, ridge, snaive
from functime.metrics import mae, smape
from functime.plotting import plot_forecasts

# Registers the `.ts` namespace of the feature extractors on polars expressions.
import functime.feature_extractors  # noqa: F401

pl.Config.set_tbl_width_chars(200)
pl.Config.set_tbl_rows(20)

# The snapshot, `$S2_IPC_DIR` or `ipcs` in the replay directory when written by swarmy.
IPC_DIR = s2.ipc_dir()

# One row per player of every game with its tracker player_id, race letter and result. It reads
# unit_born to map the player names to the player ids, so it is collected once.
players = s2.players(IPC_DIR).collect()
games_1v1 = s2.one_vs_one(IPC_DIR).collect()
players.head()

ext_fs_id,player_id,player_name,player,race,player_result,player_team_id,title,ext_datetime,ext_protocol_version
u64,u8,str,str,str,str,u8,str,datetime[ns],u32
0,1,"""&lt;MkersA&gt;<sp/>Rocck""","""Rocck""","""Z""","""Win""",0,"""King's Cove LE""",2019-08-13 22:39:44.775573,75689
1,1,"""&lt;Xanz&gt;<sp/>NoVA""","""NoVA""","""Z""","""Win""",0,"""Cyber Forest LE""",2019-08-13 22:47:30.061092,75689
2,1,"""DPX""","""DPX""","""P""","""Win""",0,"""Turbo Cruise '84 LE""",2019-08-13 23:03:22.778817,75689
0,2,"""&lt;Mealen&gt;<sp/>Doombringer""","""Doombringer""","""Z""","""Loss""",1,"""King's Cove LE""",2019-08-13 22:39:44.775573,75689
32,2,"""&lt;FxB&gt;<sp/>sandboX""","""sandboX""","""Z""","""Loss""",1,"""Cyber Forest LE""",2019-08-15 07:45:22.260022,75689


## 1. Panel data

functime works on **panel data**: a long data frame with three columns, in this order:

| entity | time | value |
|---|---|---|
| the series, i.e. a matchup, an upgrade, a player in a game | a date or an integer | the target |

Many series are forecast at once, one per entity. The time must be regular (every week, every
minute), the forecasters create the lags (`y[t-1]`, `y[t-2]`...) from it.

The games are 1v1 here. For the race balance the matchup is written with the races sorted, `PvT`,
`PvZ`, `TvZ`, and the win rate is the one of the first race, so 0.55 in PvT means the Protoss won
55% of the games.

In [2]:
balance = (
    games_1v1.lazy()
    .filter(
        (pl.col("race") < pl.col("opponent_race"))
        & pl.col("race").is_in(["P", "T"])
        & pl.col("opponent_race").is_in(["T", "Z"])
    )
    .with_columns(pl.col("ext_datetime").dt.truncate("1w").alias("time"))
    .group_by("matchup", "time")
    .agg(
        (pl.col("player_result") == "Win").mean().alias("win_rate"),
        pl.len().alias("games"),
    )
    .filter(pl.col("time") >= pl.datetime(2022, 1, 1))
    .sort("matchup", "time")
    .collect()
)
# Weeks without games are added with the win rate of the previous week, functime needs regular
# weeks. With few games a week the win rate is noisy, more replays make better series.
y_balance = (
    balance.upsample("time", every="1w", group_by="matchup")
    .with_columns(pl.col("matchup").forward_fill())
    .with_columns(pl.col("win_rate").forward_fill().over("matchup"), pl.col("games").fill_null(0))
)
y_balance.group_by("matchup").agg(
    pl.len().alias("weeks"),
    (pl.col("games") == 0).sum().alias("weeks_without_games"),
    pl.col("games").sum(),
    pl.col("win_rate").mean().alias("mean_win_rate"),
).sort("matchup")

matchup,weeks,weeks_without_games,games,mean_win_rate
str,u32,u32,u32,f64
"""PvT""",242,47,3383,0.454525
"""PvZ""",242,40,3539,0.541799
"""TvZ""",242,35,3424,0.509214


## 2. Forecasting the race balance

`train_test_split` keeps the last weeks of each series for the test. Each forecaster is a
function, configured first, then called with the train data and the forecast horizon `fh`:

- `naive`: the last value repeats, the baseline every model must beat.
- `snaive(sp=4)`: seasonal naive, the value of 4 weeks before.
- `linear_model(lags=8)` / `ridge(lags=8)`: a (regularized) linear regression on the last 8 weeks,
  one model for all the matchups.

`mae` is the mean absolute error, in win rate points here.

In [3]:
y = y_balance.select("matchup", "time", "win_rate")
fh = 12
y_train, y_test = train_test_split(test_size=fh, eager=True)(y)
forecasts = {
    "naive": naive(freq="1w")(y=y_train, fh=fh),
    "snaive 4 weeks": snaive(freq="1w", sp=4)(y=y_train, fh=fh),
    "linear 8 lags": linear_model(freq="1w", lags=8)(y=y_train, fh=fh),
    "ridge 8 lags": ridge(freq="1w", lags=8)(y=y_train, fh=fh),
}
pl.DataFrame(
    [
        {"model": name, "mae": mae(y_true=y_test, y_pred=y_pred)["mae"].mean()}
        for name, y_pred in forecasts.items()
    ]
).sort("mae")

/home/seb/build/s2-polars-pf/.venv/lib/python3.14/site-packages/functime/forecasting/naive.py:55: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  .select(pl.col(y_pred_vals.columns[-1]).alias(target).list.tail(fh))


model,mae
str,f64
"""naive""",0.103545
"""linear 8 lags""",0.145816
"""snaive 4 weeks""",0.156743
"""ridge 8 lags""",0.165005


The naive forecast is hard to beat: the balance is close to 50% and moves with the patches and the
metagame, not with the previous weeks. That is a result too, a model that does not beat `naive`
has nothing to say about the next weeks.

`rank_point_forecasts` ranks the entities by how well they were forecast, `plot_forecasts` draws
the actual series with the forecast.

In [4]:
rank_point_forecasts(y_true=y_test, y_pred=forecasts["ridge 8 lags"])

matchup,smape
str,f64
"""PvZ""",0.091369
"""TvZ""",0.179013
"""PvT""",0.184947


In [5]:
plot_forecasts(
    y_true=y.filter(pl.col("time") >= pl.datetime(2025, 6, 1)),
    y_pred=forecasts["ridge 8 lags"],
    num_cols=3,
    layout_kwargs={"height": 400, "width": 1200},
)

### Cross-validation

A single train/test split depends on the weeks chosen. `expanding_window_split` makes several
splits, each training on all the weeks before its test weeks: the spread of the errors shows how
stable the model is.

In [6]:
splits = expanding_window_split(test_size=4, n_splits=5, step_size=4, eager=True)(y)
rows = []
for fold, (fold_train, fold_test) in splits.items():
    for name, model in {"naive": naive(freq="1w"), "ridge 8 lags": ridge(freq="1w", lags=8)}.items():
        y_pred = model(y=fold_train, fh=4)
        rows.append({"fold": fold, "model": name, "mae": mae(y_true=fold_test, y_pred=y_pred)["mae"].mean()})
pl.DataFrame(rows).pivot(on="model", index="fold", values="mae")

/home/seb/build/s2-polars-pf/.venv/lib/python3.14/site-packages/functime/forecasting/naive.py:55: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  .select(pl.col(y_pred_vals.columns[-1]).alias(target).list.tail(fh))
/home/seb/build/s2-polars-pf/.venv/lib/python3.14/site-packages/functime/forecasting/naive.py:55: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  .select(pl.col(y_pred_vals.columns[-1]).alias(target).list.tail(fh))
/home/seb/build/s2-polars-pf/.venv/lib/python3.14/site-packages/functime/forecasting/naive.py:55: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potent

fold,naive,ridge 8 lags
i64,f64,f64
0,0.214239,0.12373
1,0.095663,0.04343
2,0.082222,0.142675
3,0.0,0.073586
4,0.141949,0.153724


A fold where `naive` has no error at all is a warning about the data, not a great model: the weeks
without games were filled with the win rate of the previous week, exactly what `naive` predicts.
The filled weeks flatter the naive forecast. Try two-week or monthly buckets, or only the weeks
with at least 10 games, and compare again.

## 3. Feature extraction: are the upgrades researched earlier?

The feature extractors summarize a whole series in one number, they are polars expressions under
the `.ts` namespace, so they work in `group_by().agg()`:

- `linear_trend()`: the slope, intercept and residuals of a line through the series, the slope is
  in seconds per month here.
- `mean_change()`: the average change from one month to the next.
- `absolute_sum_of_changes()`: how much the series moves, noisy series have large values.
- `longest_streak_below_mean()`: the longest run of months faster than the average.

The series is the median game time (Faster speed) at which each upgrade finished, per month, the
months with fewer than 20 games are dropped.

In [7]:
UPGRADES = [
    "zerglingmovementspeed",
    "GlialReconstitution",
    "ZergMissileWeaponsLevel1",
    "Stimpack",
    "ShieldWall",
    "TerranInfantryWeaponsLevel1",
    "WarpGateResearch",
    "BlinkTech",
    "Charge",
    "ProtossGroundWeaponsLevel1",
]
upgrade_months = (
    s2.scan(IPC_DIR, "upgrades")
    .filter(pl.col("name").is_in(UPGRADES) & (pl.col("ext_replay_loop") > 0))
    .join(
        players.lazy().select("ext_fs_id", "player_id", "ext_datetime"),
        on=["ext_fs_id", "player_id"],
    )
    .with_columns(s2.game_seconds(), pl.col("ext_datetime").dt.truncate("1mo").alias("time"))
    .group_by("name", "time")
    .agg(pl.col("game_seconds").median().alias("seconds"), pl.len().alias("games"))
    .filter(pl.col("games") >= 20)
    .sort("name", "time")
    .collect()
)
upgrade_months.group_by("name").agg(
    pl.col("seconds").ts.linear_trend().alias("trend"),
    pl.col("seconds").ts.mean_change().alias("mean_change"),
    pl.col("seconds").ts.absolute_sum_of_changes().alias("absolute_changes"),
    pl.col("seconds").ts.longest_streak_below_mean().alias("longest_months_below_mean"),
    pl.len().alias("months"),
).unnest("trend").sort("slope")

name,slope,intercept,rss,mean_change,absolute_changes,longest_months_below_mean,months
str,f64,f64,f64,f64,f64,u32,u32
"""BlinkTech""",-1.121699,442.798171,108047.562662,-0.320163,2845.558036,14,68
"""Charge""",-0.512957,487.679399,105945.099514,0.451759,2007.991071,8,68
"""ShieldWall""",-0.498913,465.091954,93785.771781,0.147516,1609.375,5,70
"""ProtossGroundWeaponsLevel1""",-0.367535,505.309348,47900.409071,0.435924,1783.035714,4,69
"""Stimpack""",-0.181232,405.023238,20705.32953,-0.324793,1229.196429,6,70
"""TerranInfantryWeaponsLevel1""",-0.077851,477.522582,256312.408892,-0.127459,2074.017857,9,70
"""WarpGateResearch""",-0.076241,238.31559,2616.464564,0.227096,464.598214,4,70
"""GlialReconstitution""",0.108453,457.379957,322436.711799,-0.446747,2659.21875,14,71
"""ZergMissileWeaponsLevel1""",1.672413,462.307728,148440.776522,3.114881,1935.089286,24,76


A negative slope means the upgrade finishes earlier every month. The intercept is the time at the
first month. Compare the slope with `absolute_changes`: a small slope with large changes is a series
that moves with the patches rather than a trend, plot it to see:

In [8]:
import plotly.express as px

px.line(
    upgrade_months.filter(pl.col("name").is_in(["zerglingmovementspeed", "Stimpack", "WarpGateResearch"])),
    x="time",
    y="seconds",
    color="name",
    title="Median time the upgrade finished, per month",
)

## 4. A global model across games: the workers of each minute

Here every **(game, player)** is an entity and the time is the **game minute**, an integer
(`freq="1i"`). The workers of minutes 7 to 10 are forecast from minutes 1 to 6. A single linear model
learns from every game at once, a *global* model, which is where functime shines compared to fitting
one model per series.

In [9]:
workers = (
    s2.scan(IPC_DIR, "stats")
    .with_columns((s2.game_seconds() // 60).cast(pl.Int64).alias("minute"))
    .filter(pl.col("minute").is_between(1, 10))
    .group_by("ext_fs_id", "player_id", "minute")
    .agg(pl.col("workers_active_count").sort_by("ext_replay_loop").last().cast(pl.Float64).alias("workers"))
    .join(
        players.lazy().select("ext_fs_id", "player_id", "race", "ext_datetime"),
        on=["ext_fs_id", "player_id"],
    )
    # Only the games that lasted the 10 minutes, the series must have the same length.
    .filter(pl.len().over("ext_fs_id", "player_id") == 10)
    .filter(pl.col("ext_datetime") >= pl.datetime(2025, 1, 1))
    .with_columns(pl.format("{}-{}", "ext_fs_id", "player_id").alias("game_player"))
    .sort("game_player", "minute")
    .select("game_player", "minute", "workers")
    .collect()
)
w_train, w_test = train_test_split(test_size=4, eager=True)(workers)
rows = []
for name, model in {
    "naive": naive(freq="1i"),
    "linear 3 lags": linear_model(freq="1i", lags=3),
    "knn 3 lags": knn(freq="1i", lags=3),
}.items():
    y_pred = model(y=w_train, fh=4)
    rows.append(
        {
            "model": name,
            "mae": mae(y_true=w_test, y_pred=y_pred)["mae"].mean(),
            # SMAPE is 0/0 when a player has no workers left, those games are skipped.
            "smape": smape(y_true=w_test, y_pred=y_pred)["smape"].drop_nans().mean(),
        }
    )
print(workers["game_player"].n_unique(), "series")
pl.DataFrame(rows).sort("mae")

/home/seb/build/s2-polars-pf/.venv/lib/python3.14/site-packages/functime/forecasting/naive.py:55: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  .select(pl.col(y_pred_vals.columns[-1]).alias(target).list.tail(fh))


4384 series


model,mae,smape
str,f64,f64
"""naive""",9.875456,0.092279
"""linear 3 lags""",12.428808,0.114305
"""knn 3 lags""",12.624806,0.114363


The linear model learns that the workers keep growing in the early game, the naive forecast assumes
they stop. With `X=` the forecasters also take exogenous features, i.e. the race or the matchup as
categorical columns, try adding them.

## 5. Features of the in-game curves: winners vs losers

The army value (minerals + vespene of the army alive, sampled every 10 seconds by the PlayerStats
events) of each player in the first 10 minutes, summarized with feature extractors:

- `slope`: how fast the army grows.
- `streak_above_mean`: the longest run of samples above the player's average army.
- `first_max_at`: where the army peaks, 1.0 is the last sample.
- `peaks`: how often the army drops, a lost fight or a trade.
- `complexity`: `cid_ce`, how jagged the curve is (NaN for a flat curve, they are dropped).

In [10]:
army_features = (
    s2.scan(IPC_DIR, "stats")
    .with_columns(
        s2.game_seconds(),
        (pl.col("minerals_used_current_army") + pl.col("vespene_used_current_army")).cast(pl.Float64).alias("army"),
    )
    .filter(pl.col("game_seconds") <= 600)
    .group_by("ext_fs_id", "player_id")
    # The order of the rows in a group is not guaranteed, the army is sorted by time first.
    .agg(
        pl.col("army").sort_by("ext_replay_loop").ts.linear_trend().struct.field("slope"),
        pl.col("army").sort_by("ext_replay_loop").ts.longest_streak_above_mean().alias("streak_above_mean"),
        pl.col("army").sort_by("ext_replay_loop").ts.first_location_of_maximum().alias("first_max_at"),
        pl.col("army").sort_by("ext_replay_loop").ts.number_peaks(support=3).alias("peaks"),
        pl.col("army").sort_by("ext_replay_loop").ts.cid_ce(normalize=True).fill_nan(None).alias("complexity"),
        pl.len().alias("samples"),
    )
    .filter(pl.col("samples") >= 30)
    .join(
        games_1v1.lazy().select("ext_fs_id", "player_id", "player_result", "matchup"),
        on=["ext_fs_id", "player_id"],
    )
    .filter(pl.col("player_result").is_in(["Win", "Loss"]))
    .collect()
)
army_features.group_by("player_result").agg(
    pl.len().alias("players"),
    pl.col("slope", "streak_above_mean", "first_max_at", "peaks", "complexity").mean(),
)

player_result,players,slope,streak_above_mean,first_max_at,peaks,complexity
str,u32,f64,f64,f64,f64,f64
"""Win""",24926,56.486761,28.677124,0.929286,1.261574,1.382326
"""Loss""",24997,46.174802,29.07949,0.889331,1.401808,1.778069


The winners grow their army faster and lose it less often in the first 10 minutes. The same
features per matchup show where it matters most, and they are inputs for a classifier that predicts
the winner at 10:00 (i.e. scikit-learn's `LogisticRegression`):

In [11]:
army_features.lazy().group_by("matchup", "player_result").agg(
    pl.len().alias("players"), pl.col("slope", "peaks").mean()
).sort("matchup", "player_result").collect()

matchup,player_result,players,slope,peaks
str,str,u32,f64,f64
"""?v?""","""Loss""",1,0.0,0.0
"""?v?""","""Win""",1,14.948195,1.0
"""PvP""","""Loss""",2991,45.145934,0.992979
"""PvP""","""Win""",2983,56.16169,0.843446
"""PvT""","""Loss""",2816,46.033892,1.212713
"""PvT""","""Win""",2577,58.386611,1.035312
"""PvZ""","""Loss""",3074,54.506904,1.061483
"""PvZ""","""Win""",2937,63.817389,0.906027
"""TvP""","""Loss""",2584,46.960539,1.458978


## Ideas to continue

- Forecast the games per week of a map to see when it leaves the map pool.
- `functime.preprocessing.detrend` and `diff` on the upgrade series before forecasting them.
- The income curve (`minerals_collection_rate`) instead of the army, and the workers lost.
- `functime.evaluation.rank_residuals` to find the games whose workers the model did not expect,
  i.e. an early harassment.